# 🔍 Model Comparison and Evaluation

## Overview
This notebook evaluates and compares the performance of different model versions across various sampling techniques and datasets. We'll analyze:
- 📊 Standard model performance
- ⚖️ Balanced model results
- 🎯 Parameter-tuned balanced model metrics

## 🛠️ Setup Environment
First, we'll set up our environment and import necessary libraries for model evaluation and visualization.

## 📚 Import Libraries <a name="import-libraries"></a>

In [1]:
import pandas as pd

import sys
sys.path.append('../../')
from utils.eval_plots import EvalPlots
from utils.model_performance_report import ModelPerformanceReport

## 📥 Load Data and Models <a name="load"></a>

In [2]:
holdout = pd.read_parquet('../../data/holdout_data.parquet')
oot = pd.read_parquet('../../data/oot_data.parquet')
train = pd.read_parquet('../../data/train_data.parquet')
metadata_columns = ['trans_date_trans_time','gender','street']

train_X = train.drop(columns=['is_fraud']+metadata_columns)
train_y = train['is_fraud']

holdout_X = holdout.drop(columns=['is_fraud']+metadata_columns)[train_X.columns]
holdout_y = holdout['is_fraud']

oot_X = oot.drop(columns=['is_fraud']+metadata_columns)[train_X.columns]
oot_y = oot['is_fraud']




In [3]:
score_standard_model = pd.read_pickle('../../models/score_balanced_model.pkl')
score_balanced_model = pd.read_pickle('../../models/score_balanced_model.pkl')
score_balanced_parameter_model = pd.read_pickle('../../models/score_balanced_parameter_model.pkl')


In [4]:
report_class = ModelPerformanceReport(train_X,train_y,holdout_X,holdout_y,oot_X,oot_y)
eval_plots = EvalPlots()

## 🔍 Compare Models

Analyze how model stability changes between the three versions:
- 📊 Standard model performance
- ⚖️ Balanced model results
- 🎯 Parameter-tuned model metrics

In [ ]:
# Compare models results

# Generate reports for standard model
standard_report = report_class.produce_report(score_standard_model)
standard_pr_auc_report = report_class.produce_pr_auc_report(score_standard_model)

# Generate reports for balanced model
balanced_report = report_class.produce_report(score_balanced_model)
balanced_pr_auc_report = report_class.produce_pr_auc_report(score_balanced_model)

# Generate reports for balanced parameter tuned model
balanced_param_report = report_class.produce_report(score_balanced_parameter_model)
balanced_param_pr_auc_report = report_class.produce_pr_auc_report(score_balanced_parameter_model)

# Combine reports into a single DataFrame for comparison
comparison_df = pd.concat([standard_report.add_suffix('_standard'), balanced_report.add_suffix('_balanced'), balanced_param_report.add_suffix('_tuned')], axis=1)
#comparison_df.columns = ['Standard Model', 'Balanced Model', 'Balanced Parameter Tuned Model']

# Display the comparison DataFrame
comparison_df

## 📈 Compare Performance Metrics

- 📊 Comprehensive metric comparison
- 📈 Performance across datasets
- 🎯 Model stability analysis
- ⚖️ Balanced vs. unbalanced results

In [ ]:
# Compare models results

# Generate reports for standard model
standard_report = report_class.produce_report(score_standard_model)
standard_pr_auc_report = report_class.produce_pr_auc_report(score_standard_model)

# Generate reports for balanced model
balanced_report = report_class.produce_report(score_balanced_model)
balanced_pr_auc_report = report_class.produce_pr_auc_report(score_balanced_model)

# Generate reports for balanced parameter tuned model
balanced_param_report = report_class.produce_report(score_balanced_parameter_model)
balanced_param_pr_auc_report = report_class.produce_pr_auc_report(score_balanced_parameter_model)

# Combine reports into a single DataFrame for comparison
comparison_df = pd.concat([standard_report.add_suffix('_standard'), balanced_report.add_suffix('_balanced'), balanced_param_report.add_suffix('_tuned')], axis=1)
#comparison_df.columns = ['Standard Model', 'Balanced Model', 'Balanced Parameter Tuned Model']

# Display the comparison DataFrame
comparison_df

In [7]:
plot_df =standard_report
plot_df['model'] = 'standard'
plot_df = pd.concat([plot_df,balanced_report])
plot_df['model'] = plot_df.model.fillna('balanced')
plot_df = pd.concat([plot_df,balanced_param_report])
plot_df['model'] = plot_df.model.fillna('balanced_param')
plot_df.reset_index(inplace=True, names='metric')
plot_df = pd.melt(plot_df, id_vars=['metric', 'model'], value_vars=['train', 'holdout', 'oot'])

In [ ]:
plot_df

📊 Visualize Performance Comparison

Interactive visualizations for:
- 📈 Metric trends across models
- 📊 Performance comparison charts
- 🔍 PR-AUC analysis
- 🎯 Model selection insights

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

app = Dash(__name__)

app.layout = html.Div([
    html.H4("Analysis of the ML model's results using scoring metrics"),
    html.P("Select metric:"),
    dcc.Dropdown(
        id='dropdown',
        options=['accuracy', 'precision', 'recall', 'f1'],
        value='precision',
        clearable=False
    ),
    dcc.Graph(id="graph"),
])


@app.callback(
    Output("graph", "figure"), 
    Input('dropdown', "value"))

def train_and_display(metric):

    fig = px.line(plot_df[plot_df.metric==metric], y='value', x='variable', color='model', markers=True)

    return fig


app.run_server(debug=True)

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

app = Dash(__name__)
MODELS = {'standard': score_standard_model, 'balanced': score_balanced_model, 'balanced_param': score_balanced_parameter_model}
app.layout = html.Div([
    html.H4("Analysis of the ML model's results using scoring metrics"),
    html.P("Select metric:"),
    dcc.Dropdown(
        id='dropdown',
        options= ['standard', 'balanced', 'balanced_param'],
        value='balanced_param',
        clearable=False
    ),
    dcc.Graph(id="graph"),
])


@app.callback(
    Output("graph", "figure"), 
    Input('dropdown', "value"))

def train_and_display(model):

    return report_class.produce_pr_auc_report(MODELS[model])


app.run_server(debug=True)